<a href="https://colab.research.google.com/github/lswoodentoys/research_paper/blob/main/h%E1%BB%87_s%E1%BB%91_t%C6%B0%C6%A1ng_quan_G177_vs_UVA340.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import pandas as pd

file_path = '/content/drive/MyDrive/Python_study/datasets/spectral_data.csv'
df = pd.read_csv(file_path)

print("Dataset shape:", df.shape)
display(df.head())

Dataset shape: (66, 3)


,Wavelength_nm,I_G177,I_UVA340
0,295,-,-
1,296,-,0.01
2,297,-,0.01
3,298,-,0.01
4,299,-,0.01


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# =====================================================
# 1. ĐỌC DỮ LIỆU PHỔ
# =====================================================

# File CSV chứa 3 cột:
# Wavelength_nm, I_G177, I_UVA340

df = pd.read_csv("spectral_data.csv")

print("Dữ liệu ban đầu:")
print(df.head())


# =====================================================
# 2. THIẾT LẬP KHOẢNG BƯỚC SÓNG
# =====================================================

lambda_min = 295  # Bước sóng giới hạn dưới (nm)
lambda_max = 360  # Bước sóng giới hạn trên (nm)


# =====================================================
# 3. LỌC VÀ ĐỒNG BỘ DỮ LIỆU
# =====================================================

df = df[
    (df["Wavelength_nm"] >= lambda_min) &
    (df["Wavelength_nm"] <= lambda_max)
].copy()

# Chuyển dữ liệu thành dạng số
columns = ["Wavelength_nm", "I_G177", "I_UVA340"]

for col in columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Loại bỏ dữ liệu không hợp lệ
df = df.dropna(subset=columns)

# Sắp xếp theo bước sóng
df = df.sort_values("Wavelength_nm")

# Nếu có bước sóng trùng nhau, tính giá trị trung bình
df = df.groupby(
    "Wavelength_nm",
    as_index=False
)[["I_G177", "I_UVA340"]].mean()

# Kiểm tra dữ liệu tham chiếu
if (df["I_G177"] <= 0).any():
    raise ValueError(
        "I_G177 phải lớn hơn 0 tại mọi bước sóng được tính."
    )

if len(df) < 2:
    raise ValueError(
        "Không đủ dữ liệu trong khoảng bước sóng đã chọn."
    )


# =====================================================
# 4. TÍNH SAI KHÁC PHỔ TẠI TỪNG BƯỚC SÓNG
# =====================================================

df["Difference"] = (
    df["I_G177"] - df["I_UVA340"]
)

df["Squared_Difference"] = (
    df["Difference"] ** 2
)

df["Chi_Square_Term"] = (
    df["Squared_Difference"] / df["I_G177"]
)


# =====================================================
# 5. TÍNH HỆ SỐ THEO CÔNG THỨC TRONG HÌNH
# =====================================================

# Tổng các thành phần chi-square
chi_square_sum = df["Chi_Square_Term"].sum()

# Số khoảng bước sóng theo công thức
denominator = lambda_max - lambda_min

# Sai khác phổ
standard_deviation = np.sqrt(
    chi_square_sum / denominator
)

# Hệ số tương đồng phổ
R = 1 - standard_deviation


# =====================================================
# 6. HIỂN THỊ KẾT QUẢ
# =====================================================

print("\n===== KẾT QUẢ TÍNH TOÁN =====")

print(f"Khoảng bước sóng: "
      f"{lambda_min} - {lambda_max} nm")

print(f"Số điểm dữ liệu: {len(df)}")

print(f"Tổng chi-square: {chi_square_sum:.8f}")

print(f"Giá trị căn bậc hai: "
      f"{standard_deviation:.8f}")

print(f"Hệ số R: {R:.8f}")


# =====================================================
# 7. XUẤT BẢNG KẾT QUẢ
# =====================================================

df.to_csv(
    "spectral_calculation_results.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nĐã xuất bảng kết quả:")
print("spectral_calculation_results.csv")


# =====================================================
# 8. VẼ BIỂU ĐỒ SO SÁNH PHỔ
# =====================================================

plt.figure(figsize=(10, 6))

plt.plot(
    df["Wavelength_nm"],
    df["I_G177"],
    label="ASTM G177",
    linewidth=2
)

plt.plot(
    df["Wavelength_nm"],
    df["I_UVA340"],
    label="UVA-340 Lamp",
    linewidth=2
)

plt.xlabel("Wavelength (nm)")
plt.ylabel("Spectral Irradiance")

plt.title(
    f"Spectral Comparison ({lambda_min}-{lambda_max} nm)"
)

plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()

plt.savefig(
    "spectral_comparison.png",
    dpi=300
)

plt.show()

In [ ]:

import numpy as np
import pandas as pd


def calculate_spectral_similarity(
    reference_file,
    lamp_file,
    lambda_min=295,
    lambda_max=360
):
    # Đọc hai phổ
    ref = pd.read_csv(reference_file)
    lamp = pd.read_csv(lamp_file)

    # Chuẩn hóa tên cột
    ref = ref.rename(columns={
        "Wavelength_nm": "Wavelength",
        "I_G177": "I_reference"
    })

    lamp = lamp.rename(columns={
        "Wavelength_nm": "Wavelength",
        "I_UVA340": "I_lamp"
    })

    # Lọc khoảng bước sóng
    ref = ref[
        (ref["Wavelength"] >= lambda_min) &
        (ref["Wavelength"] <= lambda_max)
    ].copy()

    lamp = lamp[
        (lamp["Wavelength"] >= lambda_min) &
        (lamp["Wavelength"] <= lambda_max)
    ].copy()

    # Loại bỏ bước sóng trùng nhau
    ref = ref.groupby(
        "Wavelength",
        as_index=False
    )["I_reference"].mean()

    lamp = lamp.groupby(
        "Wavelength",
        as_index=False
    )["I_lamp"].mean()

    # Tạo lưới bước sóng 1 nm
    wavelength = np.arange(
        lambda_min,
        lambda_max + 1,
        1
    )

    # Nội suy hai phổ về cùng lưới
    I_reference = np.interp(
        wavelength,
        ref["Wavelength"],
        ref["I_reference"]
    )

    I_lamp = np.interp(
        wavelength,
        lamp["Wavelength"],
        lamp["I_lamp"]
    )

    # Kiểm tra dữ liệu
    if np.any(I_reference <= 0):
        raise ValueError(
            "Phổ tham chiếu phải lớn hơn 0."
        )

    # Tính tổng chi-square
    chi_square = np.sum(
        (I_reference - I_lamp) ** 2
        / I_reference
    )

    # Tính R
    R = 1 - np.sqrt(
        chi_square / (lambda_max - lambda_min)
    )

    # Bảng kết quả
    result = pd.DataFrame({
        "Wavelength_nm": wavelength,
        "I_G177": I_reference,
        "I_UVA340": I_lamp,
        "Chi_Square_Term": (
            (I_reference - I_lamp) ** 2
            / I_reference
        )
    })

    return R, result


# Chạy tính toán
R, result = calculate_spectral_similarity(
    reference_file="ASTM_G177.csv",
    lamp_file="UVA340.csv",
    lambda_min=295,
    lambda_max=360
)

print(f"Spectral similarity coefficient R = {R:.6f}")

result.to_excel(
    "spectral_similarity_results.xlsx",
    index=False
)